<a href="https://colab.research.google.com/github/truelyfei/machine_learning/blob/main/my_choice2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# !unzip -l evaluate-main.zip
!unzip evaluate-main.zip

Archive:  evaluate-main.zip
55f1bc6e072b05c2d9db1589a07e20f38902b1ec
   creating: evaluate-main/
   creating: evaluate-main/.github/
   creating: evaluate-main/.github/hub/
  inflating: evaluate-main/.github/hub/push_evaluations_to_hub.py  
 extracting: evaluate-main/.github/hub/requirements.txt  
   creating: evaluate-main/.github/workflows/
  inflating: evaluate-main/.github/workflows/build_documentation.yml  
  inflating: evaluate-main/.github/workflows/build_pr_documentation.yml  
  inflating: evaluate-main/.github/workflows/ci.yml  
  inflating: evaluate-main/.github/workflows/delete_doc_comment.yml  
  inflating: evaluate-main/.github/workflows/python-release.yml  
  inflating: evaluate-main/.github/workflows/trufflehog.yml  
  inflating: evaluate-main/.github/workflows/update_spaces.yml  
  inflating: evaluate-main/.gitignore  
  inflating: evaluate-main/AUTHORS   
  inflating: evaluate-main/CODE_OF_CONDUCT.md  
  inflating: evaluate-main/CONTRIBUTING.md  
  inflating: evaluate-

In [5]:
!apt-get install unrar -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


In [6]:
!unrar x c3.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from c3.rar

Creating    c3                                                        OK
Extracting  c3/dataset_dict.json                                           0%  OK 
Creating    c3/test                                                   OK
Extracting  c3/test/data-00000-of-00001.arrow                              1%  OK 
Extracting  c3/test/dataset_info.json                                      1%  OK 
Extracting  c3/test/state.json                                             1%  OK 
Creating    c3/train                                                  OK
Extracting  c3/train/cache-2dc7279edc7c5646.arrow                          1%  OK 
Extracting  c3/train/cache-36b78ab287705b5c.arrow                          1%  OK 
Extracting  c3/train/cache-9505106a025af666.arrow                         24%  OK 
Extracting  c3/train/cache-c2041e9127

In [7]:
"""
多项选择任务 - C3数据集训练脚本
"""

import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

In [8]:
# 基础安装
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [9]:
import evaluate

# 查看版本
print(evaluate.__version__)

# 列出可用的评估指标
print(evaluate.list_evaluation_modules()[:10])  # 显示前10个

0.4.6
['lvwerra/test', 'angelina-wang/directional_bias_amplification', 'cpllab/syntaxgym', 'lvwerra/bary_score', 'hack/test_metric', 'yzha/ctc_eval', 'codeparrot/apps_metric', 'mfumanelli/geometric_mean', 'daiyizheng/valid', 'erntkn/dice_coefficient']


In [9]:
# import evaluate

In [10]:
# ==================== Step1: 加载数据集 ====================

def load_dataset(data_path="../data/c3/"):
    """加载C3数据集"""
    c3 = DatasetDict.load_from_disk(data_path)
    print("原始数据集结构:")
    print(c3)

    # 移除test集
    c3.pop("test")
    print("\n移除test集后:")
    print(c3)

    return c3

In [11]:
# ==================== Step8: 主训练流程 ====================

"""主函数：执行完整的训练流程"""

print("=" * 50)
print("Step1: 加载数据集")
print("=" * 50)
c3 = load_dataset("./data/c3/")

Step1: 加载数据集
原始数据集结构:
DatasetDict({
    test: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 1625
    })
    train: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 11869
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 3816
    })
})

移除test集后:
DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 11869
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'choice', 'answer'],
        num_rows: 3816
    })
})


In [12]:
# ==================== Step2: 数据预处理 ====================

def process_function(examples, tokenizer, max_length=256):
    """
    数据预处理函数
    将每个样本的4个选项分别与context和question组合，然后tokenize
    """
    # examples ["context", "question", "choice", "answer"]
    all_tokenized = []
    labels = []

    for idx in range(len(examples["context"])):
        # 将对话列表拼接成字符串
        ctx = "\n".join(examples["context"][idx]) #文本
        question = examples["question"][idx] #文本
        choices = examples["choice"][idx] #列表

        # 确保每个样本都有4个选项（不足4个时用"不知道"填充）
        padded_choices = choices.copy()
        while len(padded_choices) < 4:
            padded_choices.append("不知道") # 不确定,unknown 都可以

        # 为每个选项创建一对(context, question+choice)
        contexts = [ctx] * 4 #sentence1
        question_choices = [question + " " + choice for choice in padded_choices] #sentence2

        # tokenize
        # 1000*4 = 4000 -> 形状为 4000*256 可以放在for循环外面
        tokenized = tokenizer(
            contexts,
            question_choices,
            truncation="only_first", #第一句话可能比较长，只对第一句话contexts 截断
            max_length=max_length,
            padding="max_length"
        )

        # 将4个选项的tokenized结果组合成一个样本
        # 1000 * 4 * 256
        # sample = {k: [v[i: i+4] for i in range(0, len(v), 4)] for k, v in tokenized.items()}可以放在for循环外面
        sample = {k: [v[i] for i in range(4)] for k, v in tokenized.items()}
        all_tokenized.append(sample)

        # 找到正确答案的索引
        labels.append(choices.index(examples["answer"][idx])) #choices.index返回列表某个元素的索引号

    # 合并所有样本
    result = {k: [item[k] for item in all_tokenized] for k in all_tokenized[0].keys()}
    result["labels"] = labels

    return result

In [1]:
!pip install modelscope

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 49.6 MB/s eta 0:00:00


In [2]:
from modelscope.hub.snapshot_download import snapshot_download

In [3]:
snapshot_download(model_id="dienstag/chinese-macbert-base", cache_dir='./models/')

2026-03-28 10:56:33,311 - modelscope - INFO - Got 11 files, start to download ...


Processing 11 items:   0%|          | 0.00/11.0 [00:00<?, ?it/s]

2026-03-28 10:58:29,171 - modelscope - INFO - Download model 'dienstag/chinese-macbert-base' successfully.


'./models/dienstag/chinese-macbert-base'

In [20]:
print("Step2: 创建模型和分词器")
print("=" * 50)
model, tokenizer = create_model("./models/dienstag/chinese-macbert-base")
print(f"模型: {model.__class__.__name__}")
print(f"分词器: {tokenizer.__class__.__name__}")

Step2: 创建模型和分词器


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: ./models/dienstag/chinese-macbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if y

模型: BertForMultipleChoice
分词器: BertTokenizer


In [23]:
def preprocess_dataset(c3, tokenizer, max_length=256):
    """对整个数据集进行预处理"""
    tokenized_c3 = c3.map(
        lambda x: process_function(x, tokenizer, max_length),
        batched=True,
        remove_columns=c3["train"].column_names  # 移除原始列
    )
    return tokenized_c3

In [24]:
print("Step3: 数据预处理")
print("=" * 50)
print("正在处理训练集...")
tokenized_c3 = preprocess_dataset(c3, tokenizer)
print(f"训练集样本数: {len(tokenized_c3['train'])}")
print(f"验证集样本数: {len(tokenized_c3['validation'])}")

Step3: 数据预处理
正在处理训练集...


Map:   0%|          | 0/11869 [00:00<?, ? examples/s]

Map:   0%|          | 0/3816 [00:00<?, ? examples/s]

训练集样本数: 11869
验证集样本数: 3816


In [25]:
# 查看预处理后的数据格式
sample = tokenized_c3["train"][0]
print(f"\n预处理后的样本格式:")
for key in sample.keys():
    if key == "labels":
        print(f"  {key}: {sample[key]}")
    else:
        print(f"  {key}: shape {np.array(sample[key]).shape}")


预处理后的样本格式:
  input_ids: shape (4, 256)
  token_type_ids: shape (4, 256)
  attention_mask: shape (4, 256)
  labels: 2


In [14]:
# ==================== Step3: 创建模型 ====================

def create_model(model_path="../models/dienstag/chinese-macbert-base"):
    """创建多项选择模型"""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForMultipleChoice.from_pretrained(model_path)

    # 确保参数是连续的（避免某些优化问题）
    for param in model.parameters():
        param.data = param.data.contiguous()

    return model, tokenizer

In [15]:
!cp -r evaluate-main/metrics ./

In [16]:
# ==================== Step4: 评估函数 ====================

def compute_metrics(eval_pred):
    """计算准确率"""
    accuracy = evaluate.load("./metrics/accuracy")
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [17]:
# ==================== Step5: 训练配置 ====================

def get_training_args(output_dir="./multiple_choice"):
    """配置训练参数"""
    args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_accuracy",
        fp16=True,  # 如果GPU支持混合精度训练
        disable_tqdm=False,  # 显示进度条
        # save_total_limit=2,  # 只保留最好的2个checkpoint
        # learning_rate=2e-5,  # 可以调整学习率
        # warmup_ratio=0.1,    # warmup比例
        # weight_decay=0.01,   # 权重衰减
    )
    return args

In [18]:
# ==================== Step6: 训练器创建 ====================

# def create_trainer(model, tokenizer, train_dataset, eval_dataset, args):
#     """创建Trainer对象"""
#     trainer = Trainer(
#         model=model,
#         args=args,
#         tokenizer=tokenizer,
#         train_dataset=train_dataset,
#         eval_dataset=eval_dataset,
#         compute_metrics=compute_metrics
#     )
#     return trainer
def create_trainer(model, tokenizer, train_dataset, eval_dataset, args):
    """创建Trainer对象 - transformers 5.0.0 兼容版"""
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,  # 5.0.0 中使用 processing_class 而不是 tokenizer
        compute_metrics=compute_metrics
    )
    return trainer

In [19]:
# ==================== Step7: 预测Pipeline ====================

class MultipleChoicePipeline:
    """多项选择预测Pipeline"""

    def __init__(self, model, tokenizer, max_length=256):
        self.model = model
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.device = next(model.parameters()).device #model.device

    def preprocess(self, context, question, choices):
        """预处理输入"""
        contexts = []
        question_choices = []

        for choice in choices:
            contexts.append(context)
            question_choices.append(question + " " + choice)

        # 确保有4个选项
        while len(contexts) < 4:
            contexts.append(context)
            question_choices.append(question + " " + "不知道")

        # tokenize
        inputs = self.tokenizer(
            contexts,
            question_choices,
            truncation="only_first",
            max_length=self.max_length,
            # padding="max_length", #不是一个批次，所以没有必要
            return_tensors="pt"
        )
        return inputs

    def predict(self, inputs):
        """模型预测"""
        inputs = {k: v.unsqueeze(0).to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = self.model(**inputs).logits
        return logits

    def postprocess(self, logits, choices):
        """后处理，返回预测结果"""
        prediction = torch.argmax(logits, dim=-1).cpu().item()
        return choices[prediction] if prediction < len(choices) else None

    def __call__(self, context, question, choices):
        """完整的预测流程"""
        inputs = self.preprocess(context, question, choices)
        logits = self.predict(inputs)
        result = self.postprocess(logits, choices)
        return result

In [21]:
print("Step4: 配置训练参数")
print("=" * 50)
args = get_training_args()
print(f"输出目录: {args.output_dir}")
print(f"训练轮数: {args.num_train_epochs}")
print(f"批次大小: {args.per_device_train_batch_size}")
print(f"学习率: {args.learning_rate}")

Step4: 配置训练参数
输出目录: ./multiple_choice
训练轮数: 3
批次大小: 16
学习率: 5e-05


In [34]:
# 检查transformers版本
import transformers
print(f"transformers version: {transformers.__version__}")

transformers version: 5.0.0


In [26]:
print("Step5: 创建训练器")
print("=" * 50)
trainer = create_trainer(
    model, tokenizer,
    tokenized_c3["train"],
    tokenized_c3["validation"],
    args
)

Step5: 创建训练器


In [1]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       3.9Gi       5.0Gi       2.0Mi       3.8Gi       8.5Gi
Swap:             0B          0B          0B


In [27]:
print("Step6: 开始训练")
print("=" * 50)
trainer.train()

Step6: 开始训练


Epoch,Training Loss,Validation Loss,Accuracy
1,0.979984,0.991585,0.601153
2,0.661779,0.979734,0.637317
3,0.281629,1.331650,0.640723


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2226, training_loss=0.6922343259230457, metrics={'train_runtime': 1783.043, 'train_samples_per_second': 19.97, 'train_steps_per_second': 1.248, 'total_flos': 1.873702246273229e+16, 'train_loss': 0.6922343259230457, 'epoch': 3.0})

In [28]:
print("Step8: 评估模型")
print("=" * 50)
eval_results = trainer.evaluate()
print(f"最终评估结果: {eval_results}")

Step8: 评估模型


最终评估结果: {'eval_loss': 1.331649661064148, 'eval_accuracy': 0.6407232704402516, 'eval_runtime': 57.7304, 'eval_samples_per_second': 66.1, 'eval_steps_per_second': 4.14, 'epoch': 3.0}


In [29]:
print("Step9: 测试Pipeline")
print("=" * 50)
pipe = MultipleChoicePipeline(model, tokenizer)

Step9: 测试Pipeline


In [38]:
# 测试示例
context = """从少年时代起，富兰克林就自己照顾自己的生活，他还经常为了省钱买书而饿肚子。有一天，富兰克林在路上看到一位白发老奶奶，她已经饿得走不动路了。富兰克林将心比心，将自己仅有的一块面包送给了她。老奶奶看到富兰克林也是一个穷人，不忍心收他的面包。"你吃吧，我包里有的是。"富兰克林说着拍了拍那个装满书籍的背包。老奶奶吃着面包，只见富兰克林从背包里抽出一本书，津津有味地读起来。"孩子，你怎么不吃面包呀？"老奶奶问道。富兰克林笑着回答说："读书的味道要比面包好多了！"因为没有足够的钱，购书能力有限，富兰克林只能经常借书读。他常在晚上向朋友借书，说第二天一早准时送还。然后点起一盏灯，连夜苦读，累了就用冷水浇头提神，然后坐下来继续读。第二天一早，再准时把书还给主人，从来没有违背自己的诺言。"""

question = "富兰克林为什么总是饿肚子？"
choices = ["因为他要省钱买书", "因为他不会挣钱", "因为家里太穷了", "因为他不会照顾自己"]

result = pipe(context, question, choices)
print(f"问题: {question}")
print(f"选项: {choices}")
print(f"预测结果: {result}")

问题: 富兰克林为什么总是饿肚子？
选项: ['因为他要省钱买书', '因为他不会挣钱', '因为家里太穷了', '因为他不会照顾自己']
预测结果: 因为他要省钱买书
